# Melt Gelato | BDM Capstone Final Analysis
**Part 1: Data Understanding & Quality Assessment**  

### Introduction

Before starting the sales analysis, I first examined the original
dataset to understand its structure and identify any data quality issues.

This included checking the date range, missing values, duplicate records,
transaction types and financial columns. The aim was to understand what
needed cleaning before using the data for further analysis.

## 1. Setup 


In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np
from IPython.display import display


HERE = Path.cwd()
ROOT = HERE.parent if HERE.name == "notebooks" else HERE


RAW_DIR = ROOT / "data" / "raw"
PROCESSED_DIR = ROOT / "data" / "processed"

PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

DATA_PATH = RAW_DIR / "file.csv"

assert DATA_PATH.exists(), f"File not found: {DATA_PATH.resolve()}"

print("Project root:", ROOT.resolve())
print("Dataset found:", DATA_PATH.resolve())

Project root: /Users/aakritiabhaysingh/Desktop/melt final
Dataset found: /Users/aakritiabhaysingh/Desktop/melt final/data/raw/file.csv


In [3]:
from pathlib import Path
import pandas as pd
import numpy as np
from IPython.display import display

pd.set_option("display.max_columns", 40)
pd.set_option("display.width", 130)
DATA_PATH = RAW_DIR / "file.csv"
assert DATA_PATH.exists(), f"File not found: {DATA_PATH.resolve()}"
print("Dataset:", DATA_PATH.resolve())

Dataset: /Users/aakritiabhaysingh/Desktop/melt final/data/raw/file.csv


## 2. Loading the original POS export

In [4]:
df_raw = pd.read_csv(DATA_PATH, encoding="utf-16", sep="\t", low_memory=False)
print(f"Rows: {len(df_raw):,} | Columns: {df_raw.shape[1]}")
display(df_raw.head(5))

Rows: 122,419 | Columns: 33


,Date,Time,Time Zone,Category,Item,Qty,Price Point Name,SKU,Modifiers Applied,Product Sales,Discounts,Net Sales,Tax,Gross Sales,Transaction ID,Payment ID,Device Name,Notes,Details,Event Type,Location,Dining Option,Customer ID,Customer Name,Customer Reference ID,Unit,Count,Itemisation Type,Fulfilment Note,Channel,Token,Card Brand,PAN Suffix
0,2025-11-30,22:01:14,London,Gelato,Gelato,1.0,2 Scoops,NaN,NaN,€5.50,€0.00,€5.50,€0.00,€5.50,yyIYXYm9rRBdF1B9UToPKRsOtOaZY,lQzq2VpV9UeTRFZNOvY2qHfw60aZY,iPad,NaN,https://app.squareup.com/dashboard/sales/trans...,Payment,SEAWHEEL RETAIL LIMITED,Eat In,NaN,NaN,NaN,ea,1,Physical Good,NaN,SEAWHEEL RETAIL LIMITED,QTVOEFLJUMG47BXOAGTIOHXM,NaN,NaN
1,2025-11-30,22:01:14,London,Toppings,Topping (Voided),10.0,Regular,NaN,NaN,€0.00,€0.00,€0.00,€0.00,€0.00,yyIYXYm9rRBdF1B9UToPKRsOtOaZY,lQzq2VpV9UeTRFZNOvY2qHfw60aZY,iPad,NaN,https://app.squareup.com/dashboard/sales/trans...,Payment,SEAWHEEL RETAIL LIMITED,Eat In,NaN,NaN,NaN,ea,10,Physical Good,NaN,SEAWHEEL RETAIL LIMITED,6NCH2KSDPBMHMI7AZSVHYJNW,NaN,NaN
2,2025-11-30,21:34:18,London,Gelato,Gelato,1.0,2 Scoops,NaN,NaN,€5.50,€0.00,€5.50,€0.00,€5.50,U7SzP9hHedIEs7KluFrp1mVliq9YY,PbkdAO8ReCsBTG9nW8ZWnc7i74bZY,Square Register 0657,NaN,https://app.squareup.com/dashboard/sales/trans...,Payment,SEAWHEEL RETAIL LIMITED,Eat In,NaN,NaN,NaN,ea,1,Physical Good,NaN,SEAWHEEL RETAIL LIMITED,QTVOEFLJUMG47BXOAGTIOHXM,Visa,887.0
3,2025-11-30,21:31:37,London,Milkshakes,Brownie Milkshake,1.0,Regular,NaN,NaN,€6.50,€0.00,€6.50,€0.00,€6.50,gBM6ywBiXF28ZACOIjemhcXc4wPZY,TF4hURh9nuv9jv6wRq9Wjm2XJoIZY,Square Register 0657,NaN,https://app.squareup.com/dashboard/sales/trans...,Payment,SEAWHEEL RETAIL LIMITED,Eat In,NaN,NaN,NaN,ea,1,Physical Good,NaN,SEAWHEEL RETAIL LIMITED,WF3NFLIVQM6JKROEL236LGTB,Visa,8742.0
4,2025-11-30,21:30:10,London,Milkshakes,Vanilla Milkshake,1.0,Regular,NaN,NaN,€6.00,€0.00,€6.00,€0.00,€6.00,QLfDm1LZlhva1BudHpzboQnUtz9YY,3T8gunXFmmGy9UFZlKvqkBRolYUZY,Square Register 0657,NaN,https://app.squareup.com/dashboard/sales/trans...,Payment,SEAWHEEL RETAIL LIMITED,Eat In,PS9WJSPD3HD629Z4BWHBMGPK4C,Fionn Timmons,NaN,ea,1,Physical Good,NaN,SEAWHEEL RETAIL LIMITED,BVZWU4YSLV7MULIMNME3WVUG,Visa,4358.0


## 3. Schema and metadata


In [5]:
metadata = pd.DataFrame({
    "column": df_raw.columns,
    "dtype": df_raw.dtypes.astype(str).values,
    "non_null": df_raw.notna().sum().values,
    "missing": df_raw.isna().sum().values,
    "missing_pct": (df_raw.isna().mean().values * 100).round(2),
    "unique_non_null": df_raw.nunique(dropna=True).values,
})
display(metadata)

,column,dtype,non_null,missing,missing_pct,unique_non_null
0,Date,object,122419,0,0.00,360
1,Time,object,122419,0,0.00,33959
2,Time Zone,object,122419,0,0.00,1
3,Category,object,118629,3790,3.10,35
4,Item,object,122419,0,0.00,637
5,Qty,float64,122419,0,0.00,37
6,Price Point Name,object,116299,6120,5.00,24
7,SKU,float64,0,122419,100.00,0
8,Modifiers Applied,object,426,121993,99.65,19
9,Product Sales,object,122419,0,0.00,365


## 4. Date coverage and validity


In [6]:
dates = pd.to_datetime(df_raw["Date"], errors="coerce")
print("First date:", dates.min())
print("Last date:", dates.max())
print("Invalid/missing dates:", dates.isna().sum())
print("Distinct recorded dates:", dates.nunique())
print("Days in inclusive calendar span:", (dates.max() - dates.min()).days + 1 if dates.notna().any() else "N/A")

First date: 2024-12-01 00:00:00
Last date: 2025-11-30 00:00:00
Invalid/missing dates: 0
Distinct recorded dates: 360
Days in inclusive calendar span: 365


### Note on Missing Dates

The dataset covers December 2024 to November 2025, but sales were
recorded on only 360 of the 365 calendar days.

The five dates without records were 25–26 December 2024 and
1–2 and 24 January 2025.

Some dates fall around Christmas and New Year, but the data does not
confirm whether the business was closed. Therefore, these dates
will not automatically be treated as days with zero sales.

## 5. Missing value assessment


In [7]:
missing = metadata.loc[metadata["missing"] > 0, ["column", "missing", "missing_pct"]].sort_values("missing_pct", ascending=False)
display(missing)
print("Completely empty columns:", metadata.loc[metadata["missing_pct"] == 100, "column"].tolist())

,column,missing,missing_pct
7,SKU,122419,100.00
24,Customer Reference ID,122413,100.00
28,Fulfilment Note,122419,100.00
8,Modifiers Applied,121993,99.65
17,Notes,121117,98.94
23,Customer Name,110377,90.16
22,Customer ID,109368,89.34
21,Dining Option,26897,21.97
31,Card Brand,24451,19.97
32,PAN Suffix,24451,19.97


Completely empty columns: ['SKU', 'Customer Reference ID', 'Fulfilment Note']


## 6. Duplicate assessment
An order can contain several item rows with the same transaction ID. Therefore, **repeated transaction IDs are not automatically duplicates**. Investigating exact row duplicates separately from repeated transaction-item combinations.

In [8]:
print("Exact duplicate rows:", int(df_raw.duplicated().sum()))
print("Non-null transaction IDs:", int(df_raw["Transaction ID"].notna().sum()))
print("Unique non-null transaction IDs:", df_raw["Transaction ID"].nunique())
keys = ["Date", "Time", "Transaction ID", "Item", "Qty"]
possible = df_raw.duplicated(subset=keys, keep=False)
print("Rows sharing date/time/transaction/item/quantity:", int(possible.sum()))
display(df_raw.loc[possible, keys + ["Event Type", "Gross Sales"]].head(15))

Exact duplicate rows: 8985
Non-null transaction IDs: 122419
Unique non-null transaction IDs: 59238
Rows sharing date/time/transaction/item/quantity: 16435


,Date,Time,Transaction ID,Item,Qty,Event Type,Gross Sales
15,2025-11-30,20:44:27,UdHVPtzbpOQQOQc6T8ej9RyjlUXZY,Cans Drink,1.0,Payment,€1.65
17,2025-11-30,20:44:27,UdHVPtzbpOQQOQc6T8ej9RyjlUXZY,Cans Drink,1.0,Payment,€1.65
27,2025-11-30,20:21:15,CmiEG3Wb1WArk2Brm3ayOwX1R6KZY,Gelato,1.0,Payment,€3.80
28,2025-11-30,20:21:15,CmiEG3Wb1WArk2Brm3ayOwX1R6KZY,Gelato,1.0,Payment,€5.50
65,2025-11-30,19:00:11,C2QLQFxTflqAciYBRtTZ9qbHev5YY,Gelato (Voided),2.0,Payment,€0.00
66,2025-11-30,19:00:11,C2QLQFxTflqAciYBRtTZ9qbHev5YY,Topping (Voided),1.0,Payment,€0.00
70,2025-11-30,19:00:11,C2QLQFxTflqAciYBRtTZ9qbHev5YY,Cadbury Hot Chocolate (Voided),1.0,Payment,€0.00
71,2025-11-30,19:00:11,C2QLQFxTflqAciYBRtTZ9qbHev5YY,Americano (Voided),1.0,Payment,€0.00
72,2025-11-30,19:00:11,C2QLQFxTflqAciYBRtTZ9qbHev5YY,Gelato (Voided),3.0,Payment,€0.00
73,2025-11-30,19:00:11,C2QLQFxTflqAciYBRtTZ9qbHev5YY,Gelato (Voided),1.0,Payment,€0.00


## 7. Transaction types and financial fields
Payments, refunds and voids may need different treatment. 
Examine their frequency **before** deciding what constitutes a sale. Check monetary fields as exported, without silently converting or summing them.

In [9]:
for col in ["Event Type", "Itemisation Type", "Dining Option", "Device Name"]:
    print(f"\n{col}")
    display(df_raw[col].fillna("<missing>").value_counts(dropna=False).head(15).rename_axis(col).to_frame("rows"))

money_cols = ["Product Sales", "Discounts", "Net Sales", "Tax", "Gross Sales"]
display(df_raw[money_cols].head(10))
for col in ["Qty"]:
    print(f"\n{col} numeric conversion diagnostics")
    vals = pd.to_numeric(df_raw[col], errors="coerce")
    print("Unparseable/missing:", vals.isna().sum(), "| Negative:", (vals < 0).sum(), "| Zero:", (vals == 0).sum())


Event Type


,rows
Event Type,
Payment,121575
Refund,844



Itemisation Type


,rows
Itemisation Type,
Physical Good,119875
Prepared Food and Beverage,2063
<missing>,481



Dining Option


,rows
Dining Option,
Eat In,95265
<missing>,26897
Takeaway,255
Delivery,2



Device Name


,rows
Device Name,
Square Register 0657,71259
Melt Cafe Navan,28133
iPad,17742
<missing>,4965
iPad (7),263
iPhone (39),57


,Product Sales,Discounts,Net Sales,Tax,Gross Sales
0,€5.50,€0.00,€5.50,€0.00,€5.50
1,€0.00,€0.00,€0.00,€0.00,€0.00
2,€5.50,€0.00,€5.50,€0.00,€5.50
3,€6.50,€0.00,€6.50,€0.00,€6.50
4,€6.00,€0.00,€6.00,€0.00,€6.00
5,€6.50,€0.00,€6.50,€0.00,€6.50
6,€2.56,€0.00,€2.56,€0.94,€3.50
7,€6.00,€0.00,€6.00,€0.00,€6.00
8,€6.00,€0.00,€6.00,€0.00,€6.00
9,€3.80,€0.00,€3.80,€0.00,€3.80



Qty numeric conversion diagnostics
Unparseable/missing: 0 | Negative: 844 | Zero: 0


## 8. Audit summary 

In [10]:
audit = pd.DataFrame([
    ("Source rows", len(df_raw)),
    ("Source columns", df_raw.shape[1]),
    ("Distinct dates", dates.nunique()),
    ("Invalid dates", dates.isna().sum()),
    ("Exact duplicate rows (not removed)", df_raw.duplicated().sum()),
    ("Unique non-null transaction IDs", df_raw["Transaction ID"].nunique()),
    ("Fully missing columns", (metadata["missing_pct"] == 100).sum()),
], columns=["metric", "value"])
display(audit)
print("PART 1 COMPLETE — no records have been deleted or altered.")

,metric,value
0,Source rows,122419
1,Source columns,33
2,Distinct dates,360
3,Invalid dates,0
4,Exact duplicate rows (not removed),8985
5,Unique non-null transaction IDs,59238
6,Fully missing columns,3


PART 1 COMPLETE — no records have been deleted or altered.



## Key Observations

The original dataset contains 122,419 records and 33 columns,
covering December 2024 to November 2025.

The initial checks identified missing values, repeated records and
different transaction types, including payments and refunds.

Although 8,985 exact duplicate rows were found, they were not
automatically removed because repeated items may represent genuine
purchases.

The financial columns also need to be converted into numeric values
before calculating sales.

These checks helped identify the cleaning steps required before
moving on to the actual analysis.